# Customer-Level Fraud Pattern Detection


In [0]:
%sql
-- Q4: Transactions that are an unusual multiple of the customer's own average (the 15x example from your spec)
WITH customer_avg AS (
  SELECT customer_id, AVG(amount) AS avg_amount
  FROM fintech_fraud_risk.gold.fact_transactions
  GROUP BY customer_id
)
SELECT
  f.transaction_id, f.customer_id, f.amount, c.avg_amount,
  ROUND(f.amount / c.avg_amount, 1) AS multiple_of_avg
FROM fintech_fraud_risk.gold.fact_transactions f
JOIN customer_avg c ON f.customer_id = c.customer_id
WHERE f.amount > c.avg_amount * 5   -- flag anything 5x+ their normal spend
ORDER BY multiple_of_avg DESC;
SELECT f.transaction_id, f.customer_id, f.device_id, f.transaction_timestamp, dv.first_seen_timestamp
FROM fintech_fraud_risk.gold.fact_transactions f
JOIN fintech_fraud_risk.gold.dim_device dv ON f.device_id = dv.device_id
WHERE f.transaction_timestamp <= dv.first_seen_timestamp + INTERVAL 1 DAY;

In [0]:
%sql

-- Q5: High-velocity customers — more than 10 transactions in a single day
SELECT
  f.customer_id, d.full_date, COUNT(*) AS txns_that_day
FROM fintech_fraud_risk.gold.fact_transactions f
JOIN fintech_fraud_risk.gold.dim_date d ON f.date_key = d.date_key
GROUP BY f.customer_id, d.full_date
HAVING COUNT(*) > 10
ORDER BY txns_that_day DESC;


In [0]:
%sql

-- Q6: Repeated failed transactions per customer (subquery + HAVING)
SELECT customer_id, failed_count FROM (
  SELECT customer_id, COUNT(*) AS failed_count
  FROM fintech_fraud_risk.gold.fact_transactions
  WHERE transaction_status = 'Failed'
  GROUP BY customer_id
) t
WHERE failed_count >= 3
ORDER BY failed_count DESC;


In [0]:
%sql

-- Q7: Unusual-hour transactions (11pm–5am), ranked by amount within that window
SELECT
  transaction_id, customer_id, amount, transaction_timestamp,
  RANK() OVER (ORDER BY amount DESC) AS amount_rank
FROM fintech_fraud_risk.gold.fact_transactions
WHERE HOUR(transaction_timestamp) >= 23 OR HOUR(transaction_timestamp) < 5
ORDER BY amount_rank
LIMIT 50;


In [0]:
%sql

-- Q8: New-device transactions — device first seen within 24 hours of the transaction
SELECT f.transaction_id, f.customer_id, f.device_id, f.transaction_timestamp, dv.first_seen_timestamp
FROM fintech_fraud_risk.gold.fact_transactions f
JOIN fintech_fraud_risk.gold.dim_device dv ON f.device_id = dv.device_id
WHERE f.transaction_timestamp <= dv.first_seen_timestamp + INTERVAL 1 DAY;